# Etapa 2: Selección de técnica de muestreo para la construcción de muestra inicial

**Materia:** Análisis de grandes volúmenes de datos (Gpo 10)  
**Institución:** Tecnológico de Monterrey, Posgrados  
**Equipo 12:**

- Carlos Eduardo Vega Campos (A01797803)
- Marco Emilio Jimenez Jimenez (A01797948)
- Martha Alicia Villalobos Facundo (A01840063)
- Jonathan Javier Monsalve Giraldo (A01840272)

**Profesores:** Dr. Iván Olmos Pineda, Luis Daniel Mendoza  
**Fecha:** 17 de mayo de 2026  
**Dataset:** NYC TLC Yellow Taxi Trip Records 2024-2025

## Objetivo del notebook

Construir una muestra representativa M de la población de viajes Yellow Taxi NYC mediante muestreo estratificado con calibración histórica. La Etapa 1 del proyecto caracterizó el dataset; esta etapa parte de los datos crudos, aplica limpieza basada en los hallazgos de Etapa 1, valida D contra distribuciones históricas verificables, y extrae M mediante `sampleBy` con piso mínimo por estrato. El notebook está pensado para ejecutarse de forma portable, tanto localmente como en Google Colab; todas las rutas de datos son relativas al notebook (`./data/raw`).

## 1. Configuración del entorno

Iniciamos sesión local de Spark. Configuración mínima: subir `spark.sql.debug.maxToStringFields` para evitar truncamiento de logs en agregaciones grandes.

Notebook portable: las rutas son relativas. Funciona en Ubuntu VM local y en Google Colab si previamente se instala Java.

In [1]:
# Dependencias de Python para el notebook. Idempotente.
!pip install -q pyspark findspark pandas matplotlib


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
# Solo en Google Colab: descomentar para instalar Java (la JVM que ejecuta Spark).
# Localmente con env-pyspark esta línea no es necesaria.
# !apt-get install openjdk-8-jdk-headless -qq > /dev/null

In [3]:
import findspark
findspark.init()

from pyspark.sql import SparkSession, functions as F
from pathlib import Path
import json

spark = SparkSession.builder.master("local[*]").getOrCreate()

# Sube el umbral del log de planes (default 25) para evitar WARN benignos al agregar
# múltiples columnas en una sola llamada.
spark.conf.set("spark.sql.debug.maxToStringFields", 100)

print(f"Spark versión: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/17 13:47:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark versión: 4.1.1


### 1.2 Descarga reproducible de los datos

Reusamos el patrón de Etapa 1: descarga idempotente desde el CDN público de TLC a `./data/raw/` solo si el archivo no existe. Esto mantiene el notebook autosuficiente y portable. En una máquina donde ya se descargaron los 24 parquets mensuales más el catálogo de zonas (por ejemplo, tras ejecutar el notebook de Etapa 1), todas las descargas devuelven `skip` y la celda termina en segundos.

In [4]:
import subprocess

CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
DATA_DIR = Path("data/raw")
YEARS = [2024, 2025]
LOOKUP_FILE = "taxi_zone_lookup.csv"


def download_if_missing(download_url, target_path):
    """Descarga `download_url` a `target_path` solo si `target_path` no existe.

    Devuelve un string con el estado: 'skip', 'ok' o 'error: <mensaje>'.
    """
    if target_path.exists():
        return "skip"

    target_path.parent.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ["curl", "-sSL", "-o", str(target_path), download_url],
        capture_output=True,
        timeout=900,
    )

    if result.returncode != 0:
        return f"error: curl exit {result.returncode}"

    return "ok"

In [5]:
# Parquets mensuales de viajes
for year in YEARS:
    for month in range(1, 13):
        filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
        url = f"{CDN_BASE}/trip-data/{filename}"
        target = DATA_DIR / filename
        status = download_if_missing(url, target)
        print(f"{status:>6}  {filename}")

# Tabla de referencia de zonas de taxi
url = f"{CDN_BASE}/misc/{LOOKUP_FILE}"
target = DATA_DIR / LOOKUP_FILE
status = download_if_missing(url, target)
print(f"{status:>6}  {LOOKUP_FILE}")

  skip  yellow_tripdata_2024-01.parquet
  skip  yellow_tripdata_2024-02.parquet
  skip  yellow_tripdata_2024-03.parquet
  skip  yellow_tripdata_2024-04.parquet
  skip  yellow_tripdata_2024-05.parquet
  skip  yellow_tripdata_2024-06.parquet
  skip  yellow_tripdata_2024-07.parquet
  skip  yellow_tripdata_2024-08.parquet
  skip  yellow_tripdata_2024-09.parquet
  skip  yellow_tripdata_2024-10.parquet
  skip  yellow_tripdata_2024-11.parquet
  skip  yellow_tripdata_2024-12.parquet
  skip  yellow_tripdata_2025-01.parquet
  skip  yellow_tripdata_2025-02.parquet
  skip  yellow_tripdata_2025-03.parquet
  skip  yellow_tripdata_2025-04.parquet
  skip  yellow_tripdata_2025-05.parquet
  skip  yellow_tripdata_2025-06.parquet
  skip  yellow_tripdata_2025-07.parquet
  skip  yellow_tripdata_2025-08.parquet
  skip  yellow_tripdata_2025-09.parquet
  skip  yellow_tripdata_2025-10.parquet
  skip  yellow_tripdata_2025-11.parquet
  skip  yellow_tripdata_2025-12.parquet
  skip  taxi_zone_lookup.csv


In [6]:
files = sorted(DATA_DIR.glob("*"))
total_bytes = sum(f.stat().st_size for f in files)

for f in files:
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f"{size_mb:>8.1f} MB   {f.name}")

print()
print(f"Archivos: {len(files)} (esperados: {len(YEARS) * 12 + 1})")
print(f"Tamaño total: {total_bytes / (1024 ** 3):.2f} GB")

     0.0 MB   taxi_zone_lookup.csv
    47.6 MB   yellow_tripdata_2024-01.parquet
    48.0 MB   yellow_tripdata_2024-02.parquet
    57.3 MB   yellow_tripdata_2024-03.parquet
    56.4 MB   yellow_tripdata_2024-04.parquet
    59.7 MB   yellow_tripdata_2024-05.parquet
    57.1 MB   yellow_tripdata_2024-06.parquet
    49.9 MB   yellow_tripdata_2024-07.parquet
    48.7 MB   yellow_tripdata_2024-08.parquet
    58.3 MB   yellow_tripdata_2024-09.parquet
    61.4 MB   yellow_tripdata_2024-10.parquet
    57.8 MB   yellow_tripdata_2024-11.parquet
    58.7 MB   yellow_tripdata_2024-12.parquet
    56.4 MB   yellow_tripdata_2025-01.parquet
    57.5 MB   yellow_tripdata_2025-02.parquet
    66.7 MB   yellow_tripdata_2025-03.parquet
    64.2 MB   yellow_tripdata_2025-04.parquet
    74.2 MB   yellow_tripdata_2025-05.parquet
    70.1 MB   yellow_tripdata_2025-06.parquet
    63.8 MB   yellow_tripdata_2025-07.parquet
    59.4 MB   yellow_tripdata_2025-08.parquet
    69.1 MB   yellow_tripdata_2025-09.parquet

## 2. Carga del dataset y downcast del esquema

Cargamos los 24 parquets mensuales dejando que Spark infiera los tipos nativos, con `mergeSchema=True` para conservar `cbd_congestion_fee` (columna que solo aparece en archivos a partir de 2025-01-05). Inmediatamente después aplicamos el downcast validado en Etapa 1 sección 6 con `selectExpr`: `tinyint` para enums, `smallint` para zonas, `float` para montos. El downcast reduce del orden de 72 bytes por fila respecto a los tipos por defecto (`long` para enums, `double` para montos), lo que equivale a aproximadamente 6 GB de presión de memoria evitable sobre el dataset completo.

**Por qué no pasamos un `StructType` explícito a `.schema()`:** TLC publica algunos parquets con `trip_distance` como Parquet `DOUBLE` y otros como `FLOAT`. Forzar el esquema a `FloatType` en la lectura provoca `PARQUET_COLUMN_DATA_TYPE_MISMATCH` porque Spark no aplica coerción `double → float` al nivel de read. La estrategia robusta es inferir + cast con `selectExpr`, que sí realiza la conversión al nivel de proyección. Es el mismo patrón documentado en Etapa 1.

Referencia oficial sobre `mergeSchema`: https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#schema-merging

In [7]:
parquet_paths = sorted(str(p) for p in DATA_DIR.glob("yellow_tripdata_*.parquet"))

df_native = (spark.read
    .option("mergeSchema", "true")
    .parquet(*parquet_paths))

zones = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_DIR / "taxi_zone_lookup.csv")))

print(f"Archivos parquet cargados: {len(parquet_paths)}")
print(f"Particiones del DataFrame de viajes: {df_native.rdd.getNumPartitions()}")
print(f"Zonas en el catálogo: {zones.count()}")

Archivos parquet cargados: 24
Particiones del DataFrame de viajes: 22
Zonas en el catálogo: 265


In [8]:
df_raw = df_native.selectExpr(
    "cast(VendorID as tinyint) VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "cast(passenger_count as tinyint) passenger_count",
    "cast(trip_distance as float) trip_distance",
    "cast(RatecodeID as tinyint) RatecodeID",
    "store_and_fwd_flag",
    "cast(PULocationID as smallint) PULocationID",
    "cast(DOLocationID as smallint) DOLocationID",
    "cast(payment_type as tinyint) payment_type",
    "cast(fare_amount as float) fare_amount",
    "cast(extra as float) extra",
    "cast(mta_tax as float) mta_tax",
    "cast(tip_amount as float) tip_amount",
    "cast(tolls_amount as float) tolls_amount",
    "cast(improvement_surcharge as float) improvement_surcharge",
    "cast(total_amount as float) total_amount",
    "cast(congestion_surcharge as float) congestion_surcharge",
    "cast(Airport_fee as float) Airport_fee",
    "cast(cbd_congestion_fee as float) cbd_congestion_fee",
)

print("Esquema con downcast aplicado:")
df_raw.printSchema()

Esquema con downcast aplicado:
root
 |-- VendorID: byte (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: byte (nullable = true)
 |-- trip_distance: float (nullable = true)
 |-- RatecodeID: byte (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: short (nullable = true)
 |-- DOLocationID: short (nullable = true)
 |-- payment_type: byte (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- extra: float (nullable = true)
 |-- mta_tax: float (nullable = true)
 |-- tip_amount: float (nullable = true)
 |-- tolls_amount: float (nullable = true)
 |-- improvement_surcharge: float (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- congestion_surcharge: float (nullable = true)
 |-- Airport_fee: float (nullable = true)
 |-- cbd_congestion_fee: float (nullable = true)



In [9]:
n_raw = df_raw.count()
print(f"Registros crudos en D: {n_raw:,}")

Registros crudos en D: 89,892,322


El conteo esperado es 89,892,322 registros, consistente con el total observado en Etapa 1 (sección 3) sobre los 24 archivos parquet mensuales. El catálogo de zonas debe traer 265 entradas. Cualquier diferencia indica que algún archivo no se descargó correctamente o que TLC publicó una revisión del dataset.

## 3. Limpieza pre-estratificación

La sección 8 de Etapa 1 consolidó diez problemas de calidad detectados durante el análisis exploratorio sobre D y propuso una corrección para cada uno. En esta etapa ejecutamos esas correcciones para entregar D limpio antes de estratificar y muestrear: los outliers contaminan el cálculo de probabilidades por estrato, y Moorthy (2025) demuestra que sin este paso las correlaciones estructurales colapsan (Pearson r entre distancia y tarifa cercano a cero en datos crudos, frente a valores por encima de 0.8 esperados tras limpieza).

El tratamiento se organiza en tres bloques alineados con la tabla de Etapa 1:

- **Filtros destructivos** (correcciones 1, 2, 4 y 10): eliminan filas con valores físicamente imposibles. Etapa 1 declaró como meta una pérdida combinada inferior al 1.5% del dataset. La pérdida observada se mide empíricamente y la celda de diagnóstico descompone la contribución de cada filtro para que cualquier desviación quede atribuible.
- **Imputaciones** (correcciones 3, 5, 6 y la parte aplicable de 7): rellenan nulos estructurales o valores fuera de dominio sin descartar filas. La diferencia respecto al plan de Etapa 1 está en la corrección 7: Etapa 1 sugirió no imputar las cinco columnas con nulos correlacionados al régimen Flex Fare (era pertinente para EDA); en esta Etapa 2 sí las imputamos a un valor económicamente coherente (cero para montos, código sintético para categóricas) y conservamos en paralelo la bandera `is_flex_fare` (sección 5) para que cualquier análisis posterior pueda aislar el régimen sin haber perdido los registros.
- **Banderas y manejo geográfico** (correcciones 7 y 8): la bandera `is_flex_fare` y el etiquetado de las zonas placeholder 264 y 265 se construyen en la sección 5 como parte de las variables de caracterización, no aquí.

### 3.1 Filtros destructivos

Los umbrales superiores se toman del análisis explícito de Etapa 1 sección 8. Sutileza importante respecto a la versión preliminar de esta celda: Etapa 1 #1 y #2 definieron los rangos válidos como `[0, p99.9]` **incluyendo el cero**, no `(0, p99.9]`. El valor cero corresponde a viajes cancelados o no cobrados que sí son registros válidos (no hubo cobro ni desplazamiento), distintos de los bogus que la corrección #10 ataca de forma específica: `distance = 0` **junto con** `fare > 0` (cobro sin desplazamiento, falla del odómetro). Un filtro `distance > 0` aplicado de forma blanket descartaría todos los viajes cancelados legítimos, sobre-eliminando del orden del 5% al 6% del dataset; conservar `distance = 0` y solo aplicar #10 cuando coexiste con `fare > 0` es la lectura correcta del plan de Etapa 1.

| Etapa 1 # | Columna | Regla | Justificación del umbral |
|---|---|---|---|
| 4 | `tpep_pickup_datetime` | conservar en `[2024-01-01, 2026-01-01)` | El dataset declara cubrir 2024 y 2025; los 59 registros con años 2002, 2007-2009, 2023 y 2026 son errores de reloj del medidor o contaminación cruzada de archivos (Etapa 1 sección 7.4) |
| 1 | `trip_distance` | conservar en `[0, 200]` mi | Cota física razonable: Manhattan a Montauk ida y vuelta son aproximadamente 200 millas (Etapa 1 sección 8). El máximo observado en D es 398,608 mi (16 vueltas a la Tierra), claramente bogus. `distance = 0` se conserva globalmente porque corresponde a viajes cancelados sin cobro |
| 2 | `fare_amount` | conservar en `[0, 1000]` USD | Etapa 1 documentó: la tarifa más alta razonable en NYC ronda los USD 1,000 (viajes interestatales raros). El máximo observado es USD 863,372 y el mínimo USD -2,261 (anulaciones bogus). `fare = 0` se conserva (no-cobros válidos) |
| 2 | `total_amount` | conservar en `[0, 1200]` USD | `total_amount = fare + tip + tolls + recargos`. Con `fare <= 1000`, una propina del 20% y tolls/recargos aeropuerto comunes (~50 USD), el total realista no excede USD 1,200. El máximo observado en D es USD 863,380, hereda los mismos outliers que `fare_amount` |
| 10 | `(trip_distance, fare_amount)` | descartar registros con `distance = 0` **y** `fare > 0` | Etapa 1 sección 9.4 documenta un cluster vertical visible en el scatterplot: alineación densa de puntos en la abscisa cero con tarifas hasta USD 100. Un viaje con tarifa cobrada implica desplazamiento físico; `distance = 0` con `fare > 0` señala falla del odómetro o viaje cancelado mal facturado, no un trayecto real |

Se descarta el filtro previo sobre `passenger_count`: Etapa 1 indicó imputación, no eliminación, para preservar cardinalidad (ver bloque 3.2).

In [10]:
df_filtered = (df_raw
    .filter(F.col("tpep_pickup_datetime") >= F.lit("2024-01-01"))
    .filter(F.col("tpep_pickup_datetime") < F.lit("2026-01-01"))
    .filter(F.col("trip_distance") >= 0)
    .filter(F.col("trip_distance") <= 200)
    .filter(F.col("fare_amount") >= 0)
    .filter(F.col("fare_amount") <= 1000)
    .filter(F.col("total_amount") >= 0)
    .filter(F.col("total_amount") <= 1200)
    .filter(~((F.col("trip_distance") == 0) & (F.col("fare_amount") > 0))))

n_filtered = df_filtered.count()
pct_removed = (n_raw - n_filtered) / n_raw * 100
print(f"Registros tras filtros destructivos: {n_filtered:,}")
print(f"Filas removidas: {n_raw - n_filtered:,} ({pct_removed:.2f}%)")

# La meta aspiracional de Etapa 1 era 1.5%. Si la pérdida observada la excede,
# se documenta en la celda markdown posterior al diagnóstico. El assert solo
# actúa como red de seguridad para casos catastróficos (>15%) que indicarían
# un cambio inesperado en el dataset o un bug en los filtros.
if pct_removed > 1.5:
    print(f"\nAviso: la pérdida ({pct_removed:.2f}%) supera la meta aspiracional de Etapa 1 (1.5%).")
    print("Revisar el diagnóstico por filtro en la celda siguiente para atribuir la causa.")

assert pct_removed < 15.0, f"Filtros removieron {pct_removed:.2f}% > 15%. Algo inesperado pasa con los datos."

Registros tras filtros destructivos: 84,437,138
Filas removidas: 5,455,184 (6.07%)

Aviso: la pérdida (6.07%) supera la meta aspiracional de Etapa 1 (1.5%).
Revisar el diagnóstico por filtro en la celda siguiente para atribuir la causa.


#### Diagnóstico: contribución de cada filtro a la pérdida total

La celda siguiente cuenta, para cada condición de exclusión por separado, cuántas filas elimina sobre `df_raw`. Las cuentas se suman en una sola pasada agregada para no escanear el dataset varias veces. La suma de pérdidas individuales es generalmente mayor que la pérdida combinada `pct_removed` calculada arriba porque varios filtros pueden eliminar las mismas filas (intersección). La utilidad del diagnóstico es identificar cuál filtro es el más agresivo y poder justificar si la pérdida total se aleja de la meta aspiracional del 1.5% que documentó Etapa 1.

In [11]:
filtros_exclusion = [
    ("date < 2024-01-01", F.col("tpep_pickup_datetime") < F.lit("2024-01-01")),
    ("date >= 2026-01-01", F.col("tpep_pickup_datetime") >= F.lit("2026-01-01")),
    ("trip_distance < 0", F.col("trip_distance") < 0),
    ("trip_distance > 200", F.col("trip_distance") > 200),
    ("fare_amount < 0", F.col("fare_amount") < 0),
    ("fare_amount > 1000", F.col("fare_amount") > 1000),
    ("total_amount < 0", F.col("total_amount") < 0),
    ("total_amount > 1200", F.col("total_amount") > 1200),
    ("distance=0 AND fare>0", (F.col("trip_distance") == 0) & (F.col("fare_amount") > 0)),
]

diag = df_raw.agg(*[
    F.sum(cond.cast("int")).alias(name) for name, cond in filtros_exclusion
]).first()

print(f"Total de filas en df_raw: {n_raw:,}\n")
print(f"{'Condición de exclusión':<28} {'Elimina':>12} {'% del total':>14}")
print("-" * 56)
for name, _ in filtros_exclusion:
    n_drop = diag[name] or 0
    pct = n_drop / n_raw * 100
    print(f"{name:<28} {n_drop:>12,} {pct:>13.2f}%")

Total de filas en df_raw: 89,892,322

Condición de exclusión            Elimina    % del total
--------------------------------------------------------
date < 2024-01-01                      57          0.00%
date >= 2026-01-01                      2          0.00%
trip_distance < 0                       0          0.00%
trip_distance > 200                 3,380          0.00%
fare_amount < 0                 3,579,644          3.98%
fare_amount > 1000                     91          0.00%
total_amount < 0                1,583,065          1.76%
total_amount > 1200                    77          0.00%
distance=0 AND fare>0           1,864,632          2.07%


#### Lectura del diagnóstico y decisión de aceptación

La pérdida combinada de 6.07% se descompone en tres causas dominantes, todas documentadas como destructivas en Etapa 1 sección 8:

- **`fare_amount < 0`: 3.98%** (aproximadamente 3.58 millones de registros). Corresponden a anulaciones, créditos y reversiones según Etapa 1 sección 5.2 (mínimo observado USD -2,261). Son transacciones contables, no viajes reales; conservarlas distorsionaría las medias y desviaciones estándar de tarifa y total que validamos en sección 6 contra benchmarks históricos.
- **`distance = 0` con `fare > 0`: 2.07%** (aproximadamente 1.86 millones de registros). El cluster bogus documentado en Etapa 1 sección 9.4 (alineación densa en la abscisa cero del scatterplot). Son fallas del odómetro o viajes cancelados mal facturados, no trayectos físicos.
- **`total_amount < 0`: 1.76%** (aproximadamente 1.58 millones de registros). Solapan en gran medida con `fare_amount < 0` (las reversiones contables tiran ambos campos a valores negativos simultáneamente); la pérdida combinada empírica del 6.07% es menor que la suma lineal del 7.81% precisamente por esa intersección.

Los demás filtros remueven cuotas mínimas: 59 timestamps fuera de rango (Etapa 1 sección 7.4) y outliers extremos por arriba de los topes que también son cifras de docenas a miles, no de millones.

**Decisión: aceptar la pérdida observada del 6.07%.** Tres argumentos sostienen la decisión:

1. **Calidad sobre cantidad.** Etapa 1 declaró los tres bloques dominantes como destructivos en su sección 8. Cumplir la regla de calidad es prioritario sobre minimizar el porcentaje removido. El dataset que sobrevive (84.4 millones de registros) sigue siendo más que suficiente para construir una muestra estratificada robusta de 5 millones en la sección 9.
2. **El sesgo de conservar es peor que la reducción.** Si conserváramos las anulaciones, las medias de `fare_amount` y `total_amount` quedarían sesgadas a la baja y los Pearson r contra `trip_distance` quedarían arrastrados por los valores negativos espurios; la validación histórica contra los USD 13.47 de Etapa 1 (sección 7.1) fallaría por razones contables, no muestrales. Si conserváramos el cluster `distance = 0 ∧ fare > 0`, la celda `manhattan` del estrato quedaría sobrerepresentada por ~1.86M registros falsos, distorsionando el resto del proceso de muestreo.
3. **La meta del 1.5% era aspiracional, no medida.** Etapa 1 declaró la meta sin cuantificar la frecuencia exacta de cada bloque de anomalías sobre el dataset completo. La medición empírica revela que las anulaciones representan aproximadamente el 4% del dataset, no la fracción submarginal que sería compatible con una pérdida combinada del 1.5%. La pérdida observada es un hallazgo cuantitativo de Etapa 2 sobre el dataset 2024-2025 completo, no una falla del plan original de Etapa 1.

`df_filtered` queda con 84,437,138 registros (93.93% del original) y alimenta la sección 3.2 (imputaciones).

### 3.2 Imputaciones

Tras los filtros destructivos, D queda libre de outliers físicamente imposibles pero conserva nulos estructurales y valores fuera de dominio que el muestreo posterior no puede ingerir. Esta sección los corrige sin descartar filas. El resultado es `df_clean`, el DataFrame que alimenta las secciones 4 en adelante.

Las imputaciones eligen valores económicamente coherentes con la semántica del registro, no estadísticas globales arbitrarias: para los nulos del régimen Flex Fare se imputa **cero** en montos (refleja que no hubo cargo cobrado o capturado) y un **código sintético** en categóricas distinguible de los valores oficiales TLC. La bandera `is_flex_fare = (payment_type == 0)` se construye en la sección 5 sobre este `df_clean` y permite a cualquier análisis posterior aislar el régimen sin haber perdido los registros.

| Etapa 1 # | Columna | Regla | Justificación |
|---|---|---|---|
| 3 | `passenger_count` | `NULL` o fuera de `[1, 6]` → `1` | Moda y mediana del dataset son 1 (Etapa 1 sección 7.1); imputar a la moda preserva la cardinalidad sin distorsionar la distribución, a diferencia de la media (~1.4) que no es representable en una variable entera. La capacidad regulatoria TLC es 1-6 pasajeros |
| 5, 6 | `cbd_congestion_fee` | `NULL` o `pickup < 2025-01-05` → `0.0` | El cargo Central Business District entró en vigor el 2025-01-05. Antes de esa fecha la columna no existía en el esquema TLC (46.4% nulos estructurales) o no debía cobrarse (21 registros 2024 con valor no nulo son cobros indebidos). Imputar a cero es el valor económicamente coherente con la regulación vigente en cada fecha |
| 7 | `congestion_surcharge` | `NULL` → `0.0` | Bajo el régimen Flex Fare el TPEP no captura el cargo (17.47% nulos correlacionados con Flex). Cero refleja "no hubo cargo registrado", consistente con el principio del modelo Flex de tarifa upfront sin desglose |
| 7 | `Airport_fee` | `NULL` → `0.0` | Mismo argumento. El cargo aeropuerto bajo Flex queda absorbido en la tarifa fija negociada upfront; cero es económicamente correcto para la columna `Airport_fee` específicamente |
| 7 | `RatecodeID` | `NULL` → `99` | Los códigos TLC oficiales son 1-6 (Standard, JFK, Newark, Nassau/Westchester, Negotiated, Group ride). Flex Fare no aplica RatecodeID por diseño: la tarifa se acuerda upfront sin esquema tarifario. Se asigna 99 como código sintético "no aplica RatecodeID", convención usada por TLC para "unknown" en otros datasets |
| 7 | `store_and_fwd_flag` | `NULL` → `"F"` | Los valores oficiales son `'Y'` (viaje almacenado y reenviado al servidor por falta de conexión del medidor) y `'N'` (transmisión en tiempo real). Bajo Flex el medidor no participa en la transacción, por lo que el flag no aplica. Se asigna `"F"` como código sintético de Flex, distinguible de los valores oficiales |

Tras la imputación todas las columnas listadas quedan sin nulos; la verificación posterior lo confirma. Las restantes columnas del esquema (`VendorID`, montos cobrados, timestamps) no tienen nulos estructurales documentados en Etapa 1.

In [12]:
df_clean = (df_filtered
    .withColumn(
        "passenger_count",
        F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count"))
         .otherwise(F.lit(1).cast("byte"))
    )
    .withColumn(
        "cbd_congestion_fee",
        F.when(
            F.col("cbd_congestion_fee").isNull() | (F.col("tpep_pickup_datetime") < F.lit("2025-01-05")),
            F.lit(0.0).cast("float")
        ).otherwise(F.col("cbd_congestion_fee"))
    )
    .withColumn("congestion_surcharge",
                F.coalesce(F.col("congestion_surcharge"), F.lit(0.0).cast("float")))
    .withColumn("Airport_fee",
                F.coalesce(F.col("Airport_fee"), F.lit(0.0).cast("float")))
    .withColumn("RatecodeID",
                F.coalesce(F.col("RatecodeID"), F.lit(99).cast("byte")))
    .withColumn("store_and_fwd_flag",
                F.coalesce(F.col("store_and_fwd_flag"), F.lit("F")))
)

print("Esquema tras imputaciones (tipos preservados):")
df_clean.printSchema()

Esquema tras imputaciones (tipos preservados):
root
 |-- VendorID: byte (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: byte (nullable = true)
 |-- trip_distance: float (nullable = true)
 |-- RatecodeID: byte (nullable = false)
 |-- store_and_fwd_flag: string (nullable = false)
 |-- PULocationID: short (nullable = true)
 |-- DOLocationID: short (nullable = true)
 |-- payment_type: byte (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- extra: float (nullable = true)
 |-- mta_tax: float (nullable = true)
 |-- tip_amount: float (nullable = true)
 |-- tolls_amount: float (nullable = true)
 |-- improvement_surcharge: float (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- congestion_surcharge: float (nullable = false)
 |-- Airport_fee: float (nullable = false)
 |-- cbd_congestion_fee: float (nullable = true)



In [13]:
imputed_cols = [
    "passenger_count",
    "cbd_congestion_fee",
    "congestion_surcharge",
    "Airport_fee",
    "RatecodeID",
    "store_and_fwd_flag",
]

null_row = df_clean.agg(*[
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in imputed_cols
]).first()

print("Nulos remanentes por columna tras imputación (esperado: 0 en todas):\n")
print(f"{'Columna':<22} {'Nulos':>10} {'Estado':>8}")
print("-" * 42)
for c in imputed_cols:
    n = null_row[c] or 0
    estado = "OK" if n == 0 else "FAIL"
    print(f"{c:<22} {n:>10,} {estado:>8}")

assert all((null_row[c] or 0) == 0 for c in imputed_cols), \
    "Quedan nulos en columnas imputadas; revisar lógica de imputación."

print("\nImputación completada exitosamente. Todas las columnas objetivo quedan sin nulos.")

Nulos remanentes por columna tras imputación (esperado: 0 en todas):

Columna                     Nulos   Estado
------------------------------------------
passenger_count                 0       OK
cbd_congestion_fee              0       OK
congestion_surcharge            0       OK
Airport_fee                     0       OK
RatecodeID                      0       OK
store_and_fwd_flag              0       OK

Imputación completada exitosamente. Todas las columnas objetivo quedan sin nulos.
